# Obligatorio — Taller de Tecnologías 2 (ORT)
## Agentic RAG con LangGraph sobre tickets de soporte y papers

**Integrantes**

| Nombre | N° de estudiante |
|---|---|
| Martin Bernheim | _completar_ |
| Matías Iglesias | 310264 |

Agente construido **a mano** con `StateGraph` de LangGraph. No se usa `create_agent()`, `create_react_agent()`, `ToolNode` ni `tools_condition`. El agente:

1. Responde preguntas sobre el dataset de Kaggle **Multilingual Customer Support Tickets**.
2. Responde preguntas sobre los papers de la cátedra con chunking + RAG.
3. Tiene memoria de corto plazo (checkpointer por `thread_id`) y de largo plazo (store por `user_id`).
4. Soporta varios usuarios y conversaciones en paralelo.
5. Distingue small-talk, preguntas sobre documentos o el dataset, y preguntas generales, y admite cuando no tiene información.
6. Usa tool calling. Además del retrieval, tiene una tool de acción real: estadísticas con pandas.

### Índice
1. Setup y conexión a servicios
2. Base de conocimiento: papers
3. Base de conocimiento: tickets
4. Tools
5. El grafo (agente)
6. Memoria
7. Pruebas
8. Documentación

---
# Parte 1 — Setup y conexión a servicios

Stack (ver `docs/plan-arquitectura.md`):

| Componente | Elección |
|---|---|
| LLM (generación y router) | `gpt-6-luna`: es el único modelo de chat que habilita la key del curso (nov-2026). `gpt-4o-mini` da 403 y `gpt-4.1-nano` está deprecado. |
| Embeddings | `text-embedding-3-small` (1536 dimensiones) |
| Base vectorial | Pinecone: un índice con dos namespaces, `papers` y `tickets` |
| Orquestación | LangGraph, con el grafo armado a mano |

Las API keys se leen de `.env`, que no se versiona. El notebook nunca imprime las keys.

In [ ]:
# Dependencias
# - En local: una sola vez, `pip install -r requirements.txt` en el .venv (esta celda no hace nada).
# - En Colab: esta celda instala todo sola.
# Cada vez que el notebook use una librería nueva, hay que agregarla acá Y en requirements.txt.
import sys
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    %pip install -q langgraph langchain-openai langchain-pinecone pinecone pandas pymupdf python-dotenv
else:
    print("Entorno local: dependencias desde requirements.txt")

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# El notebook puede correr desde la raíz del repo o desde notebook/: buscamos la carpeta data/ hacia arriba.
# En Colab: clonar el repo o subir data/ al runtime antes de correr esta celda.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists()), Path.cwd())

# Keys: en local salen del .env; en Colab, de los Secrets (ícono de la llave en la barra lateral)
if EN_COLAB:
    from google.colab import userdata
    for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
        os.environ[key] = userdata.get(key)
else:
    load_dotenv(ROOT / ".env")

DATA_DIR    = ROOT / "data"
PAPERS_DIR  = DATA_DIR / "papers"
DATASET_CSV = DATA_DIR / "dataset" / "dataset-tickets-multi-lang3-4k.csv"

# Modelos y Pinecone
LLM_MODEL       = "gpt-6-luna"
ROUTER_MODEL    = "gpt-6-luna"   # la key no habilita un modelo más chico; si aparece uno, se cambia acá
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM   = 1536
PINECONE_INDEX  = "obligatorio-tt2"
NS_PAPERS, NS_TICKETS = "papers", "tickets"

for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    print(f"{key:18} {'OK' if os.getenv(key) else 'FALTA en .env'}")
print("Dataset encontrado:", DATASET_CSV.exists(), "| Papers:", len(list(PAPERS_DIR.glob("*.pdf"))))

### Prueba de conexión
Antes de construir nada, verificamos que cada servicio responda. Si la key de OpenAI está vencida o es inválida, la celda lo dice claramente en vez de cortar con un traceback.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import openai

# Sin max_tokens: gpt-6-luna razona antes de responder y, con un límite chico, se gasta los tokens
# en el razonamiento y devuelve una respuesta vacía (nos pasó con max_tokens=10).

def probar(nombre, fn):
    try:
        print(f"✅ {nombre}: {fn()}")
    except openai.AuthenticationError:
        print(f"❌ {nombre}: key de OpenAI inválida o vencida")
    except openai.PermissionDeniedError:
        print(f"❌ {nombre}: la key no tiene permiso para este modelo")
    except Exception as e:
        print(f"❌ {nombre}: {type(e).__name__}: {str(e)[:150]}")

probar(LLM_MODEL,       lambda: ChatOpenAI(model=LLM_MODEL).invoke("Decí solo: ok").content)
probar(ROUTER_MODEL,    lambda: ChatOpenAI(model=ROUTER_MODEL).invoke("Decí solo: ok").content)
probar(EMBEDDING_MODEL, lambda: f"vector de {len(OpenAIEmbeddings(model=EMBEDDING_MODEL).embed_query('hola'))} dimensiones")

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
indices = pc.list_indexes().names()
print("✅ Pinecone conectado. Índices existentes:", indices or "(ninguno)")
print(f"Índice del obligatorio '{PINECONE_INDEX}':", "ya existe" if PINECONE_INDEX in indices else "se crea en la Parte 2")

---
# Parte 2 — Base de conocimiento: papers

Pipeline: **extracción → limpieza → segmentación por secciones → chunking → embeddings → Pinecone** (namespace `papers`).

### Decisiones de diseño

| Decisión | Elección | Por qué |
|---|---|---|
| Extracción | PyMuPDF (`pymupdf`) | Da el texto por página y el índice (`get_toc()`) de cada PDF. Los 6 documentos traen TOC. |
| Segmentación | Por secciones del TOC, ubicando cada título dentro de su página | Un chunk nunca mezcla secciones distintas, y la sección queda como metadata (`Chapter 3 > B-Trees`). |
| Tamaño de chunk | **800 tokens, solape de 100** (acordado con el grupo) | Equilibrio entre contexto suficiente y precisión en preguntas puntuales. |
| Corte | Por **oraciones** (nunca a mitad de frase); el solape arrastra las últimas oraciones del chunk anterior | Un splitter por caracteres corta frases y ecuaciones. Contamos tokens con `tiktoken` (`cl100k_base`, el de `text-embedding-3-small`). |
| Limpieza | Se quitan las referencias bibliográficas, los pies de página del libro (`Reliability \| 9`) y los guiones U+2010 de fin de línea; ligaduras `ﬁ`→`fi` | La bibliografía es ruido: las preguntas del usuario no son sobre ella y contamina la búsqueda semántica. |
| Metadata | `doc_id`, `doc_title`, `doc_type`, `section`, `page` (de PDF), `printed_page` (libro), `source` | Permite filtrar por documento (`buscar_documentos(query, doc_id=...)`) y citar la fuente. |
| IDs | Determinísticos: `{doc_id}-{n:05d}` | Re-correr el notebook sobrescribe en vez de duplicar. |
| Indexación | **Una sola vez**: si el documento ya está en Pinecone se saltea | No pagamos embeddings en cada corrida; las demás partes solo se conectan al índice. |

Se conserva el *abstract* de cada paper como sección propia (`Abstract`): lo usa la tool `resumir_documento` de la Parte 4.


In [ ]:
import re
import time
import unicodedata
from dataclasses import dataclass

import pandas as pd
import pymupdf
import tiktoken

ENC = tiktoken.get_encoding("cl100k_base")   # tokenizer de text-embedding-3-small
CHUNK_TOKENS, OVERLAP_TOKENS = 800, 100

# Registro de documentos: fragmento del nombre de archivo -> metadata fija.
# El doc_id es el identificador estable que se guarda en Pinecone y que usan las tools para filtrar.
DOCUMENTOS = {
    "2507.14805":       dict(doc_id="subliminal-learning",      tipo="paper", titulo="Subliminal Learning: Language Models Transmit Behavioral Traits via Hidden Signals in Data"),
    "Attention is all": dict(doc_id="attention-is-all-you-need", tipo="paper", titulo="Attention Is All You Need"),
    "Designing Data":   dict(doc_id="ddia",                     tipo="libro", titulo="Designing Data-Intensive Applications (Martin Kleppmann)"),
    "GPT Improving":    dict(doc_id="gpt-1",                    tipo="paper", titulo="Improving Language Understanding by Generative Pre-Training (GPT-1)"),
    "Few-Shot":         dict(doc_id="gpt-3",                    tipo="paper", titulo="Language Models are Few-Shot Learners (GPT-3)"),
    "Scaling Laws":     dict(doc_id="scaling-laws",             tipo="paper", titulo="Scaling Laws for Neural Language Models"),
}

# Secciones del libro sin contenido útil: se descartan
SECCIONES_IGNORADAS = re.compile(r"^(Copyright|Table of Contents|Index|About the Author|Colophon|Front matter)$", re.I)

# Pie de página del libro: "Título | 19" (impares) o "80 | Chapter 3: Título" (pares), un token por línea
PIE_IMPAR = re.compile(r"\n?[^\n]+\n\|\s*\n(\d+)\s*$")
PIE_PAR   = re.compile(r"\n?(\d+)\s*\n\|\s*\n[^\n]+\s*$")
HEADING_REFERENCIAS = re.compile(r"^\s*(References|REFERENCES|Bibliography)\s*$", re.M)

pdfs = sorted(PAPERS_DIR.glob("*.pdf"))
print(f"{len(pdfs)} PDFs en {PAPERS_DIR.relative_to(ROOT)}")


### 2.1 Extracción y limpieza por página

`limpiar_pagina` quita el pie de página (y guarda el número de página **impreso** del libro, que no coincide con el de PDF), y une las palabras cortadas a fin de línea. `norm` se aplica al texto final: ligaduras, guiones y espacios.


In [ ]:
def norm(s: str) -> str:
    """NFKC (ligaduras ﬁ -> fi), guion U+2010 -> '-' y espacios colapsados."""
    s = unicodedata.normalize("NFKC", s).replace("\u2010", "-").replace("\u2011", "-")
    return re.sub(r"\s+", " ", s).strip()


def limpiar_pagina(texto: str):
    """Devuelve (texto sin pie de página, número de página impreso o None)."""
    texto = texto.rstrip()
    impresa = None
    for patron in (PIE_IMPAR, PIE_PAR):
        m = patron.search(texto)
        if m:
            impresa, texto = int(m.group(1)), texto[: m.start()]
            break
    # palabra cortada a fin de línea con guion U+2010: "applica‐\ntions" -> "applications"
    texto = re.sub(r"\u2010\s*\n\s*", "", texto)
    return texto, impresa


def leer_pdf(path):
    doc = pymupdf.open(path)
    paginas, impresas = [], []
    for p in doc:
        t, imp = limpiar_pagina(p.get_text())
        paginas.append(t)
        impresas.append(imp)
    return doc, paginas, impresas


### 2.2 Segmentación por secciones y eliminación de referencias

El TOC solo dice en qué **página** empieza cada sección, pero varias secciones empiezan a mitad de página. Por eso `segmentar` busca el título dentro del texto de su página para cortar en el lugar exacto (si no lo encuentra, corta al inicio de la página).

`quitar_referencias` corta la bibliografía en cada sección: el heading `References` en los papers y la lista `[1] … [2] …` al final de cada capítulo del libro.


In [ ]:
@dataclass
class Seccion:
    ruta: str      # "Part I > Chapter 1 > Reliability"
    piezas: list   # [(página_pdf (1-based), texto)] ya limpio


def segmentar(doc, paginas, tipo):
    """Parte el documento en secciones usando el TOC."""
    toc = doc.get_toc()
    offsets, full = [], ""
    for t in paginas:
        offsets.append(len(full))
        full += t + "\n"
    offsets.append(len(full))

    cortes, ultimo = [], 0
    for nivel, titulo, pag in toc:
        pag = min(max(pag, 1), len(paginas))
        inicio = offsets[pag - 1]
        patron = re.escape(titulo.strip()[:40]).replace(r"\ ", r"\s+")
        m = re.search(patron, full[inicio: offsets[pag]], re.I)
        ultimo = max(inicio + (m.start() if m else 0), ultimo)   # las posiciones nunca retroceden
        cortes.append((ultimo, nivel, titulo.strip()))
    # lo que hay antes de la primera sección: título, autores y abstract (papers) o portada (libro)
    cortes.insert(0, (0, 0, "Abstract" if tipo == "paper" else "Front matter"))

    def piezas(i0, i1):
        out = []
        for k in range(len(paginas)):
            a, b = max(i0, offsets[k]), min(i1, offsets[k + 1])
            if a < b:
                out.append((k + 1, full[a:b]))
        return out

    secciones, pila = [], []
    for k, (pos, nivel, titulo) in enumerate(cortes):
        fin = cortes[k + 1][0] if k + 1 < len(cortes) else len(full)
        pila = [x for x in pila if x[0] < nivel] + ([(nivel, titulo)] if nivel > 0 else [])
        if SECCIONES_IGNORADAS.match(titulo):
            continue
        ruta = " > ".join(t for _, t in pila) if nivel > 0 else titulo   # el nivel 0 (abstract/portada) no es padre de nada
        secciones.append(Seccion(ruta, piezas(pos, fin)))
    return secciones


def quitar_referencias(sec: Seccion) -> Seccion:
    """Corta la bibliografía: heading 'References' (papers) o lista '[1] ... [2] ...' (libro)."""
    texto = "".join(t for _, t in sec.piezas)
    corte = None
    m = HEADING_REFERENCIAS.search(texto)
    if m:
        corte = m.start()
    else:
        m1 = re.search(r"^\[1\]\s", texto, re.M)
        if m1 and re.search(r"^\[2\]\s", texto[m1.end():], re.M):
            corte = m1.start()
    if corte is None:
        return sec
    piezas, acum = [], 0
    for pag, t in sec.piezas:
        if acum >= corte:
            break
        piezas.append((pag, t[: corte - acum]))
        acum += len(t)
    return Seccion(sec.ruta, piezas)


### 2.3 Chunking por oraciones con solape

`chunkear` agrupa oraciones hasta llegar a ~800 tokens. Al cerrar un chunk, el siguiente arranca con las últimas oraciones del anterior (~100 tokens) para no perder el hilo entre fragmentos. Las oraciones gigantes (tablas, código) se parten por tokens para que ningún chunk supere el límite. Los fragmentos de menos de 30 tokens (restos de títulos o figuras) se descartan.


In [ ]:
def n_tokens(s: str) -> int:
    return len(ENC.encode(s))


def oraciones(sec: Seccion):
    """[(página, oración)] con espacios normalizados; las oraciones gigantes se parten por tokens."""
    out = []
    for pag, t in sec.piezas:
        for o in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9“\"(\[])", norm(t)):
            if not o:
                continue
            toks = ENC.encode(o)
            if len(toks) <= CHUNK_TOKENS:
                out.append((pag, o))
            else:
                for i in range(0, len(toks), CHUNK_TOKENS - OVERLAP_TOKENS):
                    out.append((pag, ENC.decode(toks[i: i + CHUNK_TOKENS - OVERLAP_TOKENS])))
    return out


def chunkear(sec: Seccion):
    """[(página_de_inicio, texto)]: ~CHUNK_TOKENS por chunk, con ~OVERLAP_TOKENS de solape."""
    chunks, actual, toks = [], [], 0
    for pag, o in oraciones(sec):
        n = n_tokens(o)
        if actual and toks + n > CHUNK_TOKENS:
            chunks.append(actual)
            cola, t = [], 0
            for x in reversed(actual):                      # solape: últimas oraciones del chunk cerrado
                nx = n_tokens(x[1])
                if t >= OVERLAP_TOKENS or t + nx > OVERLAP_TOKENS * 1.5:
                    break
                cola.insert(0, x)
                t += nx
            actual, toks = cola, t
        actual.append((pag, o))
        toks += n
    if actual and (not chunks or actual != chunks[-1][-len(actual):]):
        chunks.append(actual)
    return [(c[0][0], " ".join(o for _, o in c)) for c in chunks if sum(n_tokens(o) for _, o in c) >= 30]


def procesar_documento(path):
    """PDF -> (doc_id, lista de chunks {id, text, metadata}, nº de páginas)."""
    meta = next(m for k, m in DOCUMENTOS.items() if k in path.name)
    doc, paginas, impresas = leer_pdf(path)
    chunks = []
    for sec in segmentar(doc, paginas, meta["tipo"]):
        for pag, texto in chunkear(quitar_referencias(sec)):
            md = dict(doc_id=meta["doc_id"], doc_title=meta["titulo"], doc_type=meta["tipo"],
                      section=sec.ruta, page=pag, source=path.name)
            if impresas[pag - 1] is not None:
                md["printed_page"] = impresas[pag - 1]
            chunks.append(dict(id=f"{meta['doc_id']}-{len(chunks):05d}", text=texto, metadata=md))
    return meta["doc_id"], chunks, doc.page_count


### 2.4 Procesamos los 6 documentos

Antes de gastar en embeddings miramos la tabla: cantidad de chunks por documento, tokens totales y costo estimado. `text-embedding-3-small` cuesta USD 0,02 por millón de tokens.


In [ ]:
chunks_por_doc, filas = {}, []
for pdf in pdfs:
    doc_id, chunks, n_pag = procesar_documento(pdf)
    chunks_por_doc[doc_id] = chunks
    toks = [n_tokens(c["text"]) for c in chunks]
    filas.append(dict(doc_id=doc_id, paginas=n_pag, chunks=len(chunks), tokens=sum(toks),
                      tok_min=min(toks), tok_max=max(toks)))

resumen = pd.DataFrame(filas).set_index("doc_id")
print(resumen.to_string())
total_tokens = int(resumen["tokens"].sum())
print(f"\nTOTAL: {resumen['chunks'].sum()} chunks, {total_tokens:,} tokens -> costo de embeddings ≈ USD {total_tokens / 1e6 * 0.02:.4f}")


In [ ]:
# Un chunk de ejemplo del libro, con su metadata
ejemplo = chunks_por_doc["ddia"][60]
print(ejemplo["id"], ejemplo["metadata"], sep="\n")
print("\n" + ejemplo["text"][:600] + " ...")


### 2.5 Índice en Pinecone y embeddings

Un único índice (`obligatorio-tt2`, 1536 dimensiones, métrica coseno) con dos namespaces: `papers` (esta parte) y `tickets` (Parte 3). Si el índice no existe se crea; si el documento ya está indexado, se saltea (no se vuelve a pagar).

Cada vector guarda como metadata todo lo de la tabla de decisiones **más el texto del chunk** (clave `text`), que es lo que lee el retriever en la Parte 4.


In [ ]:
from pinecone import ServerlessSpec

if PINECONE_INDEX not in pc.list_indexes().names():
    pc.create_index(name=PINECONE_INDEX, dimension=EMBEDDING_DIM, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
    while not pc.describe_index(PINECONE_INDEX).status["ready"]:
        time.sleep(1)
    print(f"Índice '{PINECONE_INDEX}' creado")
index = pc.Index(PINECONE_INDEX)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)


def ya_indexado(doc_id, n_chunks):
    """True si el primero y el último chunk del documento ya están en el namespace."""
    ids = [f"{doc_id}-00000", f"{doc_id}-{n_chunks - 1:05d}"]
    return len(index.fetch(ids=ids, namespace=NS_PAPERS).vectors) == 2


def indexar_documento(doc_id, chunks, lote=100):
    if ya_indexado(doc_id, len(chunks)):
        print(f"⏭️  {doc_id}: ya indexado ({len(chunks)} chunks), se saltea")
        return 0
    for i in range(0, len(chunks), lote):
        parte = chunks[i: i + lote]
        vectores = embeddings.embed_documents([c["text"] for c in parte])
        index.upsert(namespace=NS_PAPERS, vectors=[
            dict(id=c["id"], values=v, metadata={**c["metadata"], "text": c["text"]})
            for c, v in zip(parte, vectores)
        ])
    print(f"✅ {doc_id}: {len(chunks)} chunks indexados")
    return len(chunks)


nuevos = sum(indexar_documento(doc_id, chunks) for doc_id, chunks in chunks_por_doc.items())

esperados = int(resumen["chunks"].sum())
for _ in range(30):   # Pinecone es eventualmente consistente: esperamos a que las stats reflejen los upserts
    en_indice = index.describe_index_stats().namespaces.get(NS_PAPERS, {}).get("vector_count", 0)
    if en_indice >= esperados:
        break
    time.sleep(2)
print(f"\nNamespace '{NS_PAPERS}': {en_indice} vectores (esperados {esperados}) | indexados en esta corrida: {nuevos}")


### 2.6 Verificación: ¿recupera lo que debe?

Probamos la búsqueda semántica antes de conectarla al agente: una pregunta por documento (algunas en español sobre papers en inglés, porque los embeddings son multilingües) y una **pregunta fuera de tema** que no está en ningún documento. Las puntuaciones (similitud coseno) de esa última nos van a servir en la Parte 4 para fijar el umbral por debajo del cual el agente debe responder *"no tengo información suficiente"*.


In [ ]:
def buscar_papers_demo(pregunta, k=3, doc_id=None):
    res = index.query(vector=embeddings.embed_query(pregunta), top_k=k, namespace=NS_PAPERS,
                      include_metadata=True, filter={"doc_id": doc_id} if doc_id else None)
    print(f"\n❓ {pregunta}" + (f"   [doc_id={doc_id}]" if doc_id else ""))
    for r in res.matches:
        m = r.metadata
        pag = f"p.{int(m['page'])}" + (f" (impresa {int(m['printed_page'])})" if "printed_page" in m else "")
        print(f"   {r.score:.3f}  {m['doc_id']} · {m['section'].split(' > ')[-1][:45]} · {pag}")
        print(f"          {m['text'][:140]}...")
    return res


for q in [
    "¿Qué es el mecanismo de self-attention y para qué sirve?",
    "How does test loss scale with the number of parameters?",
    "¿Qué diferencia hay entre un B-tree y un LSM-tree?",
    "¿Cuántos parámetros tiene GPT-3 y cómo se evalúa en few-shot?",
    "¿Qué es el aprendizaje subliminal en modelos de lenguaje?",
    "receta de lasaña vegetariana",     # fuera de tema: ningún documento la contiene
]:
    buscar_papers_demo(q)

buscar_papers_demo("¿Qué es la replicación líder-seguidor?", doc_id="ddia");   # filtro por documento


### Qué queda listo para las partes siguientes

- Namespace `papers` poblado, con `text`, `doc_id`, `doc_title`, `section`, `page` en la metadata.
- `index`, `embeddings`, `NS_PAPERS` y `buscar_papers_demo` definidos: la Parte 4 los convierte en la tool `buscar_documentos(query, doc_id=None)`.
- Para `resumir_documento(doc_id)`: filtrar por `section` que empiece con `Abstract`, `Introduction` o `Conclusion`.


---
# Parte 3 — Base de conocimiento: tickets

_Pendiente._

- Cargar el CSV con pandas y limpiar nulos.
- Un ticket = un chunk (subject + body + answer).
- Metadata: `queue`, `priority`, `type`, `language`, `tags`.
- Indexar en el namespace `tickets`.

In [ ]:
# TODO

---
# Parte 4 — Tools

_Pendiente._

Funciones con `@tool` y docstrings claros, porque el LLM elige la tool leyendo la descripción.
- `buscar_tickets(query, filtros)`: búsqueda semántica.
- `estadisticas_tickets(...)`: conteos y filtros con pandas, porque los vectores no saben contar.
- `buscar_documentos(query, doc_id=None)`
- `resumir_documento(doc_id)`: trae abstract, introducción y conclusión por metadata.
- `guardar_memoria(hecho)` / `recuperar_memoria(query)`

Probar cada tool por separado antes de dársela al LLM.

In [ ]:
# TODO

---
# Parte 5 — El grafo (agente)

_Pendiente._

Grafo armado a mano, sin prebuilts:
- `State` que extiende `MessagesState` (ruta, memorias, iteraciones).
- Nodos: router → cargar_memoria → agente ⇄ ejecutar_tools → verificador → END.
- Ejecutor de tools y condición de corte escritos por nosotros, en lugar de `ToolNode` y `tools_condition`.
- Límite de iteraciones del loop.

In [ ]:
# TODO

---
# Parte 6 — Memoria

_Pendiente._

- Corto plazo: checkpointer por `thread_id` y sliding window con `trim_messages`.
- Largo plazo: store con namespace `(user_id, "memorias")`, compartido entre threads del mismo usuario.
- Demostrar memoria entre conversaciones y dos usuarios en paralelo.

In [ ]:
# TODO

---
# Parte 7 — Pruebas

_Pendiente._

Batería de preguntas:
- Dataset: semánticas y estadísticas.
- Cada paper.
- Small-talk.
- Preguntas generales.
- Multi-turno.
- Memoria de largo plazo.
- Preguntas trampa sin respuesta en el material.

In [ ]:
# TODO

---
# Parte 8 — Documentación

## 8.1 Arquitectura y justificación de decisiones técnicas
_Pendiente._

## 8.2 Estrategia de memoria
_Pendiente._

## 8.3 Prompts y prompt engineering
_Pendiente._

## 8.4 Desafíos y soluciones
_Pendiente._

## 8.5 Pruebas y resultados
_Pendiente._

## 8.6 Reflexión y experimentación opcional
_Pendiente._

## 8.7 Uso de IA generativa
_Pendiente: herramientas usadas y para qué (obligatorio)._